# Rhema Care Flow — Colab Audit Lab

Laboratório Colab para clonar, auditar e validar build do repo.

Fonte de verdade: `main` + PR pequeno + Sentinel/TMR.

Não cole tokens reais neste notebook.


In [ ]:
REPO_URL = "https://github.com/JoaoRG-lab/rhema-care-flow.git"
REPO_DIR = "rhema-care-flow"
BRANCH = "main"


In [ ]:
import os
import subprocess
import pathlib
import json
from datetime import datetime

def run(cmd, cwd=None, check=False):
    print(f"\n$ {cmd}")
    p = subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(p.stdout)
    if check and p.returncode:
        raise RuntimeError(f"Command failed with exit code {p.returncode}: {cmd}")
    return p.returncode, p.stdout

def repo_path(*parts):
    return pathlib.Path(REPO_DIR, *parts)

def normalize_git_url(url: str) -> str:
    return (url or "").strip().replace(".git", "").rstrip("/")


In [ ]:
if not os.path.exists(REPO_DIR):
    run(f"git clone {REPO_URL} {REPO_DIR}", check=True)
else:
    code, origin = run("git remote get-url origin", cwd=REPO_DIR, check=True)
    expected = normalize_git_url(REPO_URL)
    actual = normalize_git_url(origin)
    if actual != expected:
        raise RuntimeError(
            f"Existing directory {REPO_DIR!r} points to wrong origin: {actual!r}; expected {expected!r}. "
            "Remove the folder or set REPO_DIR to a clean path."
        )
    run("git fetch --all --prune", cwd=REPO_DIR, check=True)

run(f"git checkout {BRANCH}", cwd=REPO_DIR, check=True)
run(f"git pull --ff-only origin {BRANCH}", cwd=REPO_DIR, check=True)

dirty_code, dirty = run("git status --porcelain", cwd=REPO_DIR, check=True)
if dirty.strip():
    raise RuntimeError(
        "Working tree is dirty before audit. Stop and inspect local files before trusting results.\n"
        + dirty
    )

run("git status --short --branch", cwd=REPO_DIR, check=True)
run("git log -1 --oneline", cwd=REPO_DIR, check=True)


In [ ]:
run("node -v")
run("npm -v")
run("npm install --legacy-peer-deps", cwd=REPO_DIR, check=True)


In [ ]:
env_file = repo_path(".env.example")
env_text = env_file.read_text(encoding="utf-8") if env_file.exists() else ""

env_checks = {
    "has_VITE_SUPABASE_URL": "VITE_SUPABASE_URL" in env_text,
    "has_VITE_SUPABASE_PUBLISHABLE_KEY": "VITE_SUPABASE_PUBLISHABLE_KEY" in env_text,
    "deprecated_VITE_SUPABASE_ANON_KEY_absent": "VITE_SUPABASE_ANON_KEY" not in env_text,
    "documents_PERPLEXITY_API_KEY": "PERPLEXITY_API_KEY" in env_text,
    "documents_GEMINI_API_KEY": "GEMINI_API_KEY" in env_text,
}

print(json.dumps(env_checks, indent=2, ensure_ascii=False))


In [ ]:
critical_files = [
    "src/integrations/supabase/client.ts",
    "supabase/functions/ai-assistant/index.ts",
    "src/components/ai/AISiteAgentWidget2.tsx",
    "src/components/ai/AIIntegrationPanel.tsx",
    "src/router.tsx",
    "vercel.json",
    ".github/workflows/tmr-deploy.yml",
    ".github/workflows/audit-sentinel.yml",
]

critical_checks = {
    f"exists::{file}": repo_path(file).exists()
    for file in critical_files
}

print(json.dumps(critical_checks, indent=2, ensure_ascii=False))


In [ ]:
def read_file(rel):
    p = repo_path(rel)
    return p.read_text(encoding="utf-8") if p.exists() else ""

edge = read_file("supabase/functions/ai-assistant/index.ts")
panel = read_file("src/components/ai/AIIntegrationPanel.tsx")
widget = read_file("src/components/ai/AISiteAgentWidget2.tsx")
supabase_client = read_file("src/integrations/supabase/client.ts")

contract_checks = {
    "edge_returns_reply": "reply:" in edge or '"reply"' in edge,
    "edge_returns_answer": "answer" in edge,
    "panel_reads_reply": "data?.reply" in panel or "data.reply" in panel,
    "panel_sends_agent": "agent:" in panel,
    "widget_site_publico": "site_publico" in widget,
    "edge_cors_reumatismos": "reumatismos.com" in edge,
    "edge_rate_limit": "RATE_LIMIT" in edge or "rateLimitMap" in edge,
    "frontend_uses_publishable_key": "VITE_SUPABASE_PUBLISHABLE_KEY" in supabase_client,
}

print(json.dumps(contract_checks, indent=2, ensure_ascii=False))


In [ ]:
tsc_code, _ = run("npx tsc --noEmit", cwd=REPO_DIR)
lint_code, _ = run("npm run lint --if-present", cwd=REPO_DIR)
build_code, _ = run("npm run build", cwd=REPO_DIR)

build_checks = {
    "typescript_exit_code": tsc_code,
    "lint_exit_code": lint_code,
    "build_exit_code": build_code,
    "build_passed": build_code == 0,
}

print(json.dumps(build_checks, indent=2, ensure_ascii=False))


In [ ]:
integration_blockers = {
    **{k: v for k, v in env_checks.items()},
    **{k: v for k, v in critical_checks.items()},
    **{k: v for k, v in contract_checks.items()},
}

failed_integration_checks = [k for k, v in integration_blockers.items() if not v]
audit_passed = build_code == 0 and not failed_integration_checks

report = f"""# Rhema Care Flow — Colab Audit Report

Generated: {datetime.utcnow().isoformat()}Z

Branch: {BRANCH}

## Build results
- TypeScript exit code: {tsc_code}
- Lint exit code: {lint_code}
- Build exit code: {build_code}

## Integration blockers
{json.dumps(failed_integration_checks, indent=2, ensure_ascii=False)}

## Final verdict
{"PASS" if audit_passed else "FAIL"}

## Interpretation
- PASS = build passed and all env/critical/contract checks passed.
- FAIL = inspect blockers before opening or merging a PR.
- This notebook is an auxiliary lab only; GitHub Sentinel/TMR remain the source of truth.
"""

repo_path("audit_report.md").write_text(report, encoding="utf-8")
print(report)

if not audit_passed:
    raise RuntimeError("Audit failed. See blockers above.")
